# 🚛 Freight Rate Prediction Challenge — Interactive ML Notebook
### Spotter Machine Learning Engineer Assessment
**Author:** ML Engineering Candidate  
**Goal:** Train and validate a high-precision spot freight rate prediction model (`posted_rate`) on 48,000 historical loads and predict 12,000 validation loads and a 31-day December scenario.

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print('Environment initialized successfully!')

## 1. Load Data & Inspect Schemas

In [2]:
train_df = pd.read_csv('../data/train_test.csv')
val_df = pd.read_csv('../data/validation.csv')
dec_df = pd.read_csv('../data/december_chart_inputs.csv')

print(f'Train Shape: {train_df.shape}')
print(f'Val Shape:   {val_df.shape}')
print(f'Dec Shape:   {dec_df.shape}')
display(train_df.head(3))

## 2. Feature Engineering & Preprocessing
We construct **38 high-signal features** covering geospatial geometry, cyclical temporal harmonics, physical load metrics (ton-miles, weight/mile), and regularized Bayes target encodings.

In [3]:
sys.path.append('..')
from src.data_loader import build_city_coordinate_registry
from src.feature_engineering import engineer_features, apply_target_encodings, FEATURE_COLUMNS

city_coords = build_city_coordinate_registry(train_df, val_df)

val_temp = val_df.copy()
val_temp['date'] = pd.to_datetime(val_temp['date'])
val_dec = val_temp[val_temp['date'].dt.month == 12]
daily_mi_dec = val_dec.groupby(val_dec['date'].dt.date)['market_index'].mean().to_dict()
daily_qs_dec = val_dec.groupby(val_dec['date'].dt.date)['quote_signal'].mean().to_dict()
daily_signals = (daily_mi_dec, daily_qs_dec)

median_weight = float(train_df['weight'].median())
median_mi = float(train_df['market_index'].median())

train_feat = engineer_features(train_df, city_coords, None, median_weight, median_mi)
val_feat = engineer_features(val_df, city_coords, daily_signals, median_weight, median_mi)
dec_feat = engineer_features(dec_df, city_coords, daily_signals, median_weight, median_mi)

train_proc, val_proc, dec_proc = apply_target_encodings(train_feat, val_feat, dec_feat)
print(f'Engineered Feature Count: {len(FEATURE_COLUMNS)}')
print('Features:', FEATURE_COLUMNS[:10], '...')

## 3. Train Multi-Objective Stacked Ensemble (5-Fold CV)

In [4]:
from src.models import FreightRateEnsemble

ensemble = FreightRateEnsemble()
oof_preds = ensemble.fit_cv(train_proc)

mae = mean_absolute_error(train_proc['posted_rate'], oof_preds)
rmse = np.sqrt(mean_squared_error(train_proc['posted_rate'], oof_preds))
r2 = r2_score(train_proc['posted_rate'], oof_preds)
mape = np.mean(np.abs((train_proc['posted_rate'] - oof_preds) / train_proc['posted_rate'])) * 100

print(f'\nOverall 5-Fold OOF MAE:  ${mae:.2f}')
print(f'Overall 5-Fold OOF RMSE: ${rmse:.2f}')
print(f'Overall 5-Fold OOF R²:   {r2:.4f}')
print(f'Overall 5-Fold OOF MAPE: {mape:.2f}%')

## 4. Residual Diagnostics & Evaluation Visualizations

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
axes[0].scatter(train_proc['posted_rate'], oof_preds, alpha=0.25, color='#2b580c', s=12)
axes[0].plot([0, train_proc['posted_rate'].max()], [0, train_proc['posted_rate'].max()], 'r--', lw=2)
axes[0].set_title('Actual vs Out-of-Fold Predicted Rates ($)')
axes[0].set_xlabel('Actual Rate ($)')
axes[0].set_ylabel('Predicted Rate ($)')

residuals = train_proc['posted_rate'] - oof_preds
sns.histplot(residuals, kde=True, color='#d9534f', bins=50, ax=axes[1])
axes[1].set_title('Residual Error Distribution (Actual - Predicted)')
axes[1].set_xlabel('Error ($)')
axes[1].set_xlim(-1500, 1500)
plt.tight_layout()
plt.show()

## 5. Generate Submission Predictions & Verify with `score.py`

In [6]:
from src.inference import generate_and_save_predictions
import subprocess

val_sub, dec_sub = generate_and_save_predictions(val_df, dec_df, val_proc, dec_proc, ensemble)

result = subprocess.run([
    sys.executable, '../score.py',
    '--predictions', '../validation_predictions.csv',
    '--december-predictions', '../december_predictions.csv'
], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print(result.stderr)